# SVM eval timing: one recorded dataset against `svm-best`

Loads the published RBF-SVM (`training/models/svm-best/`: `SVC(C=1000, gamma=1e-05)` + `StandardScaler`) and scores **one** 1{,}440-clip CQT dataset. Purpose is wall-clock cost, not a results table: how long the 1.3\,GB pickle takes to load, how long 40 clips take to predict, then the full dataset, then a 5-dataset extrapolation.

Does not write metrics or overwrite the shipped model. Change `DATASET` in the config cell to time another dataset (`thinkpad`, `vivo`, `flow`, `thinkpad-2`, `flow-2`).

## Config

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATASET = "thinkpad"  # same 1,440-clip size as vivo / flow / thinkpad-2 / flow-2
N_PROBE = 40           # one class worth of takes, for a cheap first clock
RUN_FULL = True        # False = stop after the probe and only extrapolate


def find_training_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "features").is_dir() and (candidate / "models").is_dir():
            return candidate
        nested = candidate / "training"
        if (nested / "features").is_dir() and (nested / "models").is_dir():
            return nested
    raise FileNotFoundError(f"Could not locate training root from {start}")


HERE = Path.cwd().resolve()
TRAINING_ROOT = find_training_root(HERE)
MODEL_DIR = TRAINING_ROOT / "models" / "svm-best"
FEATURES_PATH = TRAINING_ROOT / "features" / f"{DATASET}.npz"

for p in (MODEL_DIR / "model.pkl", MODEL_DIR / "scaler.pkl", MODEL_DIR / "label_encoder.pkl", FEATURES_PATH):
    assert p.is_file(), f"missing {p}"

cfg = pd.DataFrame(
    {
        "key": ["dataset", "n_probe", "run_full", "features", "model_dir", "model_mb"],
        "value": [
            DATASET,
            N_PROBE,
            RUN_FULL,
            str(FEATURES_PATH),
            str(MODEL_DIR),
            round((MODEL_DIR / "model.pkl").stat().st_size / 1e6, 1),
        ],
    }
)
display(cfg)

,key,value
0,dataset,thinkpad
1,n_probe,40
2,run_full,True
3,features,/home/seya/code/chord-detection/training/featu...
4,model_dir,/home/seya/code/chord-detection/training/model...
5,model_mb,1348.3


## Load trained SVM

In [2]:
import time

import joblib
import pandas as pd
from IPython.display import display

t0 = time.perf_counter()
svm = joblib.load(MODEL_DIR / "model.pkl")
t_model = time.perf_counter() - t0

t0 = time.perf_counter()
scaler = joblib.load(MODEL_DIR / "scaler.pkl")
encoder = joblib.load(MODEL_DIR / "label_encoder.pkl")
t_aux = time.perf_counter() - t0

info = {
    "estimator": type(svm).__name__,
    "kernel": getattr(svm, "kernel", None),
    "C": getattr(svm, "C", None),
    "gamma": getattr(svm, "gamma", None),
    "n_support_sum": int(getattr(svm, "n_support_", [0]).sum()) if hasattr(svm, "n_support_") else None,
    "n_classes": int(len(encoder.classes_)),
    "load_model_s": round(t_model, 2),
    "load_scaler_encoder_s": round(t_aux, 2),
}
display(pd.DataFrame([info]))

/home/seya/code/chord-detection/.venv/lib/python3.10/site-datasetages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/seya/code/chord-detection/.venv/lib/python3.10/site-datasetages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/seya/code/chord-detection/.venv/lib/python3.10/site-datasetages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from

,estimator,kernel,C,gamma,n_support_sum,n_classes,load_model_s,load_scaler_encoder_s
0,SVC,rbf,1000,0.00001,4143,36,2.96,0.01


## Load features

In [3]:
import time

import numpy as np
import pandas as pd
from IPython.display import display

t0 = time.perf_counter()
with np.load(FEATURES_PATH, allow_pickle=True) as data:
    features = data["features"]
    labels = np.asarray(data["labels"]).astype(str)
t_npz = time.perf_counter() - t0

assert features.ndim == 3, features.shape
X_flat = features.reshape(features.shape[0], -1)

feat_info = pd.DataFrame(
    [
        {
            "dataset": DATASET,
            "n": int(features.shape[0]),
            "cqt": str(tuple(features.shape[1:])),
            "flat_dim": int(X_flat.shape[1]),
            "n_labels": int(len(np.unique(labels))),
            "load_npz_s": round(t_npz, 2),
        }
    ]
)
display(feat_info)

,dataset,n,cqt,flat_dim,n_labels,load_npz_s
0,thinkpad,1440,"(216, 188)",40608,36,1.4


## Time probe, then the full dataset

In [4]:
import time

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score


def time_predict(X, n):
    t0 = time.perf_counter()
    Xs = scaler.transform(X[:n])
    t_scale = time.perf_counter() - t0
    t0 = time.perf_counter()
    y_hat = svm.predict(Xs)
    t_pred = time.perf_counter() - t0
    return t_scale, t_pred, y_hat


n_probe = min(N_PROBE, len(X_flat))
t_scale_p, t_pred_p, y_probe = time_predict(X_flat, n_probe)

rows = [
    {
        "stage": f"probe n={n_probe}",
        "scale_s": round(t_scale_p, 2),
        "predict_s": round(t_pred_p, 2),
        "s_per_clip": round(t_pred_p / n_probe, 4),
        "accuracy": None,
    }
]

y_true = None
t_scale_f = t_pred_f = None
if RUN_FULL:
    t_scale_f, t_pred_f, y_hat = time_predict(X_flat, len(X_flat))
    y_true = encoder.transform(labels)
    acc = float(accuracy_score(y_true, y_hat))
    rows.append(
        {
            "stage": f"full n={len(X_flat)}",
            "scale_s": round(t_scale_f, 2),
            "predict_s": round(t_pred_f, 2),
            "s_per_clip": round(t_pred_f / len(X_flat), 4),
            "accuracy": round(acc, 4),
        }
    )
    per_clip = t_pred_f / len(X_flat)
else:
    per_clip = t_pred_p / n_probe

n_datasets = 5
n_per_dataset = 1440
extra = pd.DataFrame(
    [
        {
            "one_dataset_predict_min": round(per_clip * n_per_dataset / 60, 2),
            "five_datasets_predict_min": round(per_clip * n_per_dataset * n_datasets / 60, 2),
            "plus_one_model_load_min": round((t_model + per_clip * n_per_dataset * n_datasets) / 60, 2),
            "per_clip_s": round(per_clip, 4),
            "source": "full dataset" if RUN_FULL else f"probe n={n_probe}",
        }
    ]
)

display(pd.DataFrame(rows))
display(extra)

,stage,scale_s,predict_s,s_per_clip,accuracy
0,probe n=40,0.01,9.35,0.2338,NaN
1,full n=1440,0.43,384.05,0.2667,0.8618


,one_dataset_predict_min,five_datasets_predict_min,plus_one_model_load_min,per_clip_s,source
0,6.4,32.0,32.05,0.2667,full dataset
